In [1]:
import os
os.chdir("..")
import json

In [2]:
from transformers import AutoTokenizer

tokenizers = {
    "qwen": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-32B-Instruct"),
    "llama": AutoTokenizer.from_pretrained("nvidia/Llama-3_3-Nemotron-Super-49B-v1")
}

In [3]:
templates = [
    "cot-planning/results/blocksworld_4_blocks/{model_name}/plan_generation_po.json",
    "cot-planning/results/blocksworld_mystery_1/{model_name}/plan_generation_po.json"
]

In [4]:
model_names = [
    ["deepseek-32b", "gpt-4.1-cot", "gpt-4.1", "qwen2-32b", "qwen2-32b-instruct", "qwen2-32b-instruct-cot", "qwq-32b", "deepseek-llama-70b", "llama-3.3-70b-instruct-cot", "nemotron-v1"],
    ["qwen2.5-32b-instruct-cot", "gpt-4.1-cot", "qwq-32b-greedy", "deepseek-r1-distill-llama-70b", "llama-3_3-nemotron-super-49b-v1", "qwen2.5-32b", "llama-3.3-70b-instruct-cot"]
]

exp_tokenizers = [
    ["qwen", "qwen", "qwen", "qwen", "qwen", "qwen", "qwen", "llama", "llama", "llama"],
    ["qwen", "qwen", "qwen", "llama", "llama", "qwen", "llama"]
]

experiments = [
    "4 blocks blocksworld",
    "4 blocks mystery 1 blocksworld"
]

In [5]:
def load_results(model_name, template):
    with open(template.format(model_name=model_name), "r") as f:
        return json.load(f)

In [6]:
import numpy as np

_instances = None

for template, _model_names, experiment, exp_tokenizer in zip(templates, model_names, experiments, exp_tokenizers):
    print(experiment)
    for m, t in zip(_model_names, exp_tokenizer):
        try:
            v = load_results(m, template)
            instances = v["instances"][:300]
            tokenizer = tokenizers[t]
            gen_lengths = [len(tokenizer.encode(x["full_response"])) for x in instances]
            total = len(instances)
            success = sum(
                x["llm_correct"] for x in instances
            )
            print(f"{m}: {success} / {total}: {round(success / total, 2)}, mean tokens: {np.mean(gen_lengths)}")
            if m == "llama-3_3-nemotron-super-49b-v1":
                _instances = instances
        except Exception as e:
            print(f"Error loading results for {m}: {e}")


4 blocks blocksworld
deepseek-32b: 242 / 300: 0.81, mean tokens: 2386.9666666666667
gpt-4.1-cot: 277 / 300: 0.92, mean tokens: 555.9766666666667
gpt-4.1: 164 / 300: 0.55, mean tokens: 549.2033333333334
qwen2-32b: 62 / 300: 0.21, mean tokens: 71.41
qwen2-32b-instruct: 97 / 300: 0.32, mean tokens: 125.79
qwen2-32b-instruct-cot: 114 / 300: 0.38, mean tokens: 353.2966666666667
qwq-32b: 192 / 200: 0.96, mean tokens: 3633.32
deepseek-llama-70b: 198 / 300: 0.66, mean tokens: 2674.5433333333335
llama-3.3-70b-instruct-cot: 119 / 300: 0.4, mean tokens: 764.48
nemotron-v1: 145 / 300: 0.48, mean tokens: 1161.9833333333333
4 blocks mystery 1 blocksworld
qwen2.5-32b-instruct-cot: 1 / 300: 0.0, mean tokens: 1479.4833333333333
gpt-4.1-cot: 55 / 300: 0.18, mean tokens: 3837.35
qwq-32b-greedy: 106 / 300: 0.35, mean tokens: 16185.623333333333
deepseek-r1-distill-llama-70b: 31 / 300: 0.1, mean tokens: 10635.516666666666
llama-3_3-nemotron-super-49b-v1: 19 / 100: 0.19, mean tokens: 9200.17
qwen2.5-32b: 1 /

In [8]:
print(_instances[1]["full_response"])

<think>
Okay, let's tackle this problem step by step. First, I need to understand the initial state and the goal. The initial conditions are:

- Block A craves Block C
- Block C craves Block B
- Block D craves Block A
- Harmony is true
- Planet Block B
- Province Block D

The goal is to have:

- Block A craves Block C (already true, so we need to maintain this)
- Block B craves Block A (currently not true)
- Block C craves Block D (currently not true)

So the main tasks are to get Block B to crave Block A and Block C to crave Block D, while keeping Block A's craving for C intact.

Let me recall the actions and their effects:

1. **Attack (obj)**:
   - Requires: Province obj, Planet obj, Harmony.
   - Effects: Pain obj becomes true; Province obj, Planet obj, Harmony become false.

2. **Feast (obj from other)**:
   - Requires: Object Craves other obj, Province obj, Harmony.
   - Effects: Pain obj becomes true; Province other obj becomes true. Also, Object Craves other obj, Province obj, 